# m-risk · Motor de Análisis Comparativo de Normativas
## Sprint 1 — Embeddings + Comparación + Gaps + Clasificador normas nuevas
**Sector Acuicultura · USA · Japón · Brasil**

In [ ]:
!pip install sentence-transformers openpyxl pandas numpy -q

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import hashlib
import json
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

UMBRAL_EQUIV = 0.88
UMBRAL_PARCIAL = 0.70
PRIORIDAD = {"Acceso":1,"Sanitario":2,"Inocuidad":3,"Etiquetado":4,"Ambiental":5}

rows = [
  ["Chile","LGPA Ley 18.892","SUBPESCA","Marco legal general actividades pesqueras acuicolas. Concesiones, condiciones ambientales sanitarias.","Acceso","1989","Existente","Base regulatoria salmonera Chile"],
  ["Chile","DS 319/2002","SERNAPESCA","Reglamento proteccion control erradicacion enfermedades riesgo especies hidrobiologicas RESA.","Sanitario","2002","Existente","Regula ISA SRS EAR. Programas Sanitarios centros cultivo."],
  ["Chile","DS 345/2005","SERNAPESCA","Reglamento plagas hidrobiologicas REPLA. Control organismos afecten centros cultivo.","Sanitario","2005","Existente","Control Caligus. Complementa RESA plagas."],
  ["Chile","DS 72/2011","SERNAPESCA","Reglamento certificacion requisitos sanitarios importacion exportacion especies hidrobiologicas.","Sanitario","2011","Existente","Base Autorizacion Sanitaria embarque exportacion SERNAPESCA."],
  ["Chile","DS 49/2006","SERNAPESCA","Reglamento centros acopio faenamiento productos hidrobiologicos condiciones sanitarias proceso.","Inocuidad","2006","Existente","Aplica plantas procesadoras exportadoras. Complementa HACCP."],
  ["Chile","DS 40/2012 SEIA","MMA/SEA","Reglamento Sistema Evaluacion Impacto Ambiental. EIA DGA instalacion ampliacion centros acuicolas.","Ambiental","2012","Existente","RCA vigente obligatoria operar exportar formalmente."],
  ["Chile","Ley 21.410","SERNAPESCA","Modifica LGPA exigiendo medidas evitar deposito desechos fondos marinos planes recuperacion.","Ambiental","2022","Existente","Planes recuperacion vigentes enero 2024. Vinculada auditorias ambientales."],
  ["Chile","Autorizacion SERNAPESCA","SERNAPESCA","Autorizacion Origen Legal acredita cumplimiento normativa pesquera acuicola nacional exportador.","Acceso","Vigente","Existente","Obligatoria todo embarque exportacion junto Autorizacion Sanitaria."],
  ["USA","21 CFR Part 123","FDA","Regulacion HACCP Hazard Analysis Critical Control Points procesadores pescado mariscos consumo humano.","Inocuidad","2023","Existente","Obligatoria planta extranjera exporte productos pesca USA. Plan HACCP documentado registros verificacion."],
  ["USA","FSMA Food Safety Modernization Act","FDA","Ley modernizacion inocuidad alimentaria. Registro instalaciones controles preventivos FSVP importadores.","Inocuidad","2016","Existente","Exportador registrado FDA. Importador USA aplica FSVP verificar proveedor cumple estandares."],
  ["USA","21 CFR Part 1 Prior Notice","FDA","Notificacion previa obligatoria FDA todo alimento importado USA antes arribo puerto entrada.","Acceso","Vigente","Existente","Presentar via sistema PNSI. Sin prior notice aprobada carga no ingresa USA."],
  ["USA","FSVP Foreign Supplier Verification","FDA","Programa verificacion proveedores extranjeros. Importador USA verifica exportador cumple estandares inocuidad.","Acceso","2017","Existente","Exportador debe proveer evidencia cumplimiento auditorias certificados HACCP."],
  ["USA","21 CFR Part 101 Food Labeling","FDA","Requisitos etiquetado alimentos USA. Nombre producto ingredientes alergenos informacion nutricional ingles.","Etiquetado","2023","Existente","Salmon declara si cultivo silvestre pais origen COOL metodo produccion. Alergenos declaracion obligatoria."],
  ["USA","Country of Origin Labeling COOL","USDA-AMS","Etiquetado obligatorio pais origen pescados mariscos frescos congelados refrigerados vendidos retail USA.","Etiquetado","Vigente","Existente","Salmon chileno indica Product of Chile. Aplica nivel retail."],
  ["USA","Antibiotic Residue Tolerance","FDA","Limites maximos residuos antibioticos productos acuicultura importados. Oxitetraciclina antimicrobianos aprobados.","Inocuidad","Vigente","Existente","FDA retiene embarques residuos sobre tolerancia. Chile restricciones propias alineadas."],
  ["USA","Import Alert 16-131","FDA","Alerta importacion detencion automatica productos acuicolas paises historial residuos drogas no aprobadas.","Acceso","Vigente","Existente","Chile no en alerta actualmente. Riesgo monitorear. Activacion implica detencion embarques."],
  ["Japon","Food Sanitation Act","MHLW","Ley Sanidad Alimentaria marco regulatorio inocuidad alimentos Japon. Aditivos permitidos residuos pesticidas contaminantes.","Inocuidad","2024","Existente","Todo producto acuicultura importado debe cumplir esta ley. 2024 actualizo lista positiva materiales contacto."],
  ["Japon","Lista Positiva Materiales Contacto","MHLW","Actualizacion 2024 sustancias permitidas envases materiales contacto alimentos importados.","Inocuidad","2024","Nuevo","Envases salmon fresco congelado deben cumplir nueva lista. Revision proveedores packaging requerida."],
  ["Japon","Certificado Sanitario MHLW","MHLW","Certificado sanitario emitido autoridad competente pais exportador SERNAPESCA Chile. Requerido ingreso acuicultura Japon.","Sanitario","Vigente","Existente","Japon exige presentar aduana. Sin certificado valido SERNAPESCA carga retenida rechazada."],
  ["Japon","Inspeccion Cuarentena Puerto","MHLW/Aduana","Productos alimenticios importados sujetos inspeccion sanitaria arribo. Permiso aduanero tras aprobar controles sanitarios.","Acceso","Vigente","Existente","Importador japones no tiene resultados vigilancia previos Japon puede exigir importar muestras."],
  ["Japon","Importador Domiciliado Japon","MHLW","Solo empresa domiciliada Japon puede actuar como importador legal. Responsable etiquetado tramites MHLW Aduana.","Acceso","Vigente","Existente","Exportador chileno necesita socio importador japones. Importador responde legalmente ante MHLW."],
  ["Japon","Etiquetado JAS","CAA/MAFF","Requisitos etiquetado japones nombre producto ingredientes almacenaje consumo nombre direccion importador.","Etiquetado","Vigente","Existente","Todo etiquetado en japones. Importador japones responsable adecuacion etiquetado comercializacion."],
  ["Japon","LMR Pesticidas Medicamentos","MHLW","Sistema Positive List residuos pesticidas medicamentos veterinarios. Sustancias no listadas LMR 0.01 ppm.","Inocuidad","Vigente","Existente","Extremadamente estricto. Medicamentos veterinarios usados Chile deben tener LMR establecido Japon."],
  ["Japon","Restricciones Sustancias Clase I","MHLW","Directrices actualizadas prohíben importacion productos contengan sustancias quimicas especificadas clase I.","Inocuidad","2024","Nuevo","Verificacion insumos produccion alimento peces tratamientos no contienen sustancias lista actualizada 2024."],
  ["Brasil","IN MAPA 1/2017","MAPA/DIPOA","Instruccion Normativa habilitacion sanitaria establecimientos extranjeros exportadores productos origen animal Brasil.","Acceso","2017","Existente","Planta procesadora chilena habilitada ante DIPOA antes tramitar Licencia Importacion. Sin habilitacion no exporta."],
  ["Brasil","Decreto 9.013/2017 RIISPOA","MAPA/DIPOA","Reglamento Inspeccion Industrial Sanitaria Productos Origen Animal. Estandares calidad proceso inspeccion pescados.","Inocuidad","2020","Existente","Condiciones higienico sanitarias procesamiento. Manual Fiscalizacion Pescado Derivados MAPA."],
  ["Brasil","Licencia Importacion LI DIPOA","MAPA/DIPOA","Importacion productos origen animal requiere licencia no automatica DIPOA aprobada antes embarque.","Acceso","Vigente","Existente","Tramitar via Siscomex. DIPOA analisis documental inspeccion en puerto. Retraso genera costos almacenaje."],
  ["Brasil","RDC ANVISA 727/2022","ANVISA","Rotulado obligatorio alergenos alimentarios Brasil. Pescado alergeno declaracion obligatoria.","Etiquetado","2022","Existente","Etiquetado indica presencia pescado derivados. Aplica salmon todas presentaciones."],
  ["Brasil","IN MAPA 22/2005","MAPA","Norma etiquetado comercial productos pesca Brasil. Nombre producto forma presentacion conservacion establecimiento.","Etiquetado","2021","Existente","Etiquetado en portugues. Formato rotulado aprobado DIPOA antes primera exportacion."],
  ["Brasil","PNCR Programa Residuos","MAPA","Programa monitoreo residuos medicamentos veterinarios contaminantes productos origen animal incluyendo pescado importado.","Inocuidad","Vigente","Existente","Brasil muestreos en puerto. Residuos fuera limite lote retenido destruido reembarcado."],
  ["Brasil","Portaria MAPA 884/2023","MAPA","Programa Nacional Moluscos Bivalves Seguros MoluBiS. Establece trazabilidad que se extiende productos pesqueros.","Inocuidad","2023","Nuevo","Senal regulatoria Brasil aumentando exigencias trazabilidad productos acuicolas. Monitorear proximos ciclos."],
]

cols = ["Mercado","Norma","Organismo","Requisito","Categoria","Version","Estado","Observaciones"]
df = pd.DataFrame(rows, columns=cols)
df["texto"] = df.apply(lambda r: f"[{r['Categoria']}] {r['Norma']} {r['Organismo']}: {r['Requisito']}. {r['Observaciones']}", axis=1)
df["id"] = df.apply(lambda r: hashlib.md5(f"{r['Mercado']}_{r['Norma']}_{r['Version']}".encode()).hexdigest()[:10], axis=1)

print(f"Corpus: {len(df)} normativas")
print(df['Mercado'].value_counts().to_dict())
print(df['Estado'].value_counts().to_dict())

MODEL = "paraphrase-multilingual-mpnet-base-v2"
print(f"\nCargando modelo {MODEL}...")
model = SentenceTransformer(MODEL)
print("Modelo cargado.")

print("Generando embeddings...")
vecs = model.encode(df["texto"].tolist(), show_progress_bar=True, batch_size=16)
vecs = np.array(vecs)
print(f"Embeddings: {vecs.shape}")

def clasificar(score):
    if score >= UMBRAL_EQUIV: return "ya_cubierta","Bajo"
    elif score >= UMBRAL_PARCIAL: return "variacion","Medio"
    else: return "gap_nuevo","Alto"

def comparar(origen, destinos):
    orig = df[df['Mercado']==origen]
    resultados = []
    for _, ro in orig.iterrows():
        vo = vecs[ro.name].reshape(1,-1)
        matches = {}
        for dest in destinos:
            dd = df[(df['Mercado']==dest)&(df['Categoria']==ro['Categoria'])]
            if dd.empty: continue
            vd = vecs[dd.index]
            sc = cosine_similarity(vo, vd)[0]
            top = dd.iloc[np.argsort(sc)[::-1][:3]]
            matches[dest] = [{"norma":r['Norma'],"score":round(float(sc[i]),4),"tipo":clasificar(sc[i])[0]} for i,(_,r) in enumerate(top.iterrows())]
        resultados.append({"origen_norma":ro['Norma'],"origen_mercado":ro['Mercado'],"categoria":ro['Categoria'],"equivalentes":matches})
    return resultados

def gaps(origen, destino):
    orig = df[df['Mercado']==origen]
    dest = df[df['Mercado']==destino]
    result = []
    for _, rd in dest.iterrows():
        vd = vecs[rd.name].reshape(1,-1)
        oc = orig[orig['Categoria']==rd['Categoria']]
        sc_max = float(cosine_similarity(vd, vecs[oc.index]).max()) if not oc.empty else 0.0
        if sc_max < UMBRAL_PARCIAL:
            result.append({"norma":rd['Norma'],"categoria":rd['Categoria'],"score_max":round(sc_max,4),"prioridad":PRIORIDAD.get(rd['Categoria'],9)})
    return sorted(result, key=lambda x: x['prioridad'])

def clasificar_nuevas():
    nuevas = df[df['Estado'].isin(['Nuevo','Modificado'])]
    base = df[df['Estado']=='Existente']
    result = []
    for _, rn in nuevas.iterrows():
        vn = vecs[rn.name].reshape(1,-1)
        bc = base[(base['Mercado']==rn['Mercado'])&(base['Categoria']==rn['Categoria'])]
        sc_max = float(cosine_similarity(vn, vecs[bc.index]).max()) if not bc.empty else 0.0
        tipo, riesgo = clasificar(sc_max)
        result.append({"norma":rn['Norma'],"mercado":rn['Mercado'],"categoria":rn['Categoria'],"clasificacion":tipo,"score":round(sc_max,4),"riesgo":riesgo})
    return sorted(result, key=lambda x: PRIORIDAD.get(x['categoria'],9))

print("\n" + "="*60)
print("NORMAS NUEVAS DEL CICLO MENSUAL")
print("="*60)
emo = {"ya_cubierta":"OK","variacion":"~~","gap_nuevo":"!!"}
for r in clasificar_nuevas():
    print(f"\n{emo.get(r['clasificacion'],'?')} [{r['clasificacion'].upper()}] {r['mercado']} | {r['norma']}")
    print(f"   Categoria: {r['categoria']} | Score: {r['score']} | Riesgo: {r['riesgo']}")

for dest in ["USA","Japon","Brasil"]:
    gs = gaps("Chile", dest)
    print(f"\n{'='*60}")
    print(f"GAPS Chile -> {dest}: {len(gs)} detectados")
    print("="*60)
    for g in gs:
        print(f"  !! [{g['categoria']}] {g['norma']} | score_max={g['score_max']}")

print("\n\nSPRINT 1 COMPLETADO")
